In [ ]:
# ── Choose instrument for paper backtest ──────────────────────────────
TRADE_INSTRUMENT = 'unleveraged'   # 'unleveraged' → C40.PA  |  'leveraged' → LVC.PA
instr_df = c40 if TRADE_INSTRUMENT == 'unleveraged' else lvc
instr_label = INSTRUMENT_TICKER if TRADE_INSTRUMENT == 'unleveraged' else INSTRUMENT_LEV

# Align instrument to backtest period
bt_instr = instr_df.loc[bt_start:].copy()
bt_instr['daily_return'] = bt_instr['close'].pct_change()

# Merge composite signal (forward fill 1 day to avoid lookahead)
bt = pd.DataFrame(index=bt_instr.index)
bt['signal'] = combined['nn_gated_composite'].reindex(bt.index).shift(1).fillna(0)
bt['instr_ret'] = bt_instr['daily_return']
bt['strategy_ret'] = bt['signal'] * bt['instr_ret']

# Equity curves
bt['equity_strategy']  = (1 + bt['strategy_ret'].fillna(0)).cumprod() * INITIAL_CAPITAL
bt['equity_buyhold']   = (1 + bt['instr_ret'].fillna(0)).cumprod() * INITIAL_CAPITAL
bt['equity_cactr']     = (1 + df.loc[bt_start:, 'log_return_1d'].apply(np.exp).subtract(1)
                            .reindex(bt.index).fillna(0)).cumprod() * INITIAL_CAPITAL

# Quick stats
n_years = (bt.index[-1] - bt.index[0]).days / 365.25
cagr    = (bt['equity_strategy'].iloc[-1] / INITIAL_CAPITAL) ** (1 / n_years) - 1
sr      = bt['strategy_ret'].mean() / bt['strategy_ret'].std() * np.sqrt(252)
dd      = (bt['equity_strategy'] / bt['equity_strategy'].cummax() - 1).min()

print(f'--- Strategy ({instr_label}) ---')
print(f'  CAGR:          {cagr*100:.2f}%')
print(f'  Sharpe:        {sr:.2f}')
print(f'  Max Drawdown:  {dd*100:.2f}%')
print(f'  Exposure:      {bt["signal"].mean()*100:.1f}%')

# Chart equity curves
p_eq = Chart(height=400, watermark='Equity Curves')
p_eq.line(ts(bt['equity_strategy'] / 1000, 'value'), name=f'TKAN v3 ({instr_label})', color='#00AA00')
p_eq.line(ts(bt['equity_buyhold']  / 1000, 'value'), name=f'{instr_label} Buy & Hold',  color='#4169E1', width=1)
p_eq.line(ts(bt['equity_cactr']    / 1000, 'value'), name='CAC40 TR B&H', color='#FFA500', width=1)

p_pos = Chart(height=150, watermark='Position')
p_pos.area(ts(bt['signal'], 'value'), name='In Market', color='#228B22')

Dashboard(panes=[p_eq, p_pos], titles=['Equity Curves (k€)', 'Position']).show()